# 02 - Rejection Prediction: Feature Sets & Cross-Validation (Section 3.3)

**Physical question:** How much do XPS chemistry descriptors improve rejection prediction
beyond operating conditions and membrane geometry alone - and does the improvement survive
across articles (cross-study generalisation)?

**Model:** CatBoost (500 trees, lr=0.05, depth=6).  
Salts and dyes are trained separately (different solute feature sets).

## Feature sets

| Set | Features (salts / dyes) | Description |
|-----|------------------------|-------------|
| **A** | 6 / 4 | Filtration conditions + ionic/molecular solute properties |
| **B** | 9 / 7 | A + membrane geometry (thickness, flake size, dry d-spacing) |
| **C** | 12 / 10 | B + XPS bond fractions (C–C, C–O, C=O) |
| **D** | 14 / 12 | C + surface properties (Zeta, WCA) |


## CV hierarchy

| Strategy | Leakage allowed | Interpretation |
|----------|-----------------|----------------|
| **Random** | Same membrane in train+test | Optimistic lower bound on error |
| **GroupKFold** (membrane) | Same membrane isolated | Within-article, different membranes |
| **LOAO** | Entire article held out | Cross-article / cross-study generalisation |

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
%matplotlib inline

from catboost import CatBoostRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import KFold

DATA_FILE = os.path.join('..', 'go_filtration_data.csv')

TARGET    = 'Rejection_%'
MIN_TRAIN = 5
N_SEEDS   = 50
K_FOLDS   = 5
SEED      = 42

# Feature groups
FILT     = ['pressure_bar', 'solution_concentration_M']
SALT_SOL = ['cation_charge', 'anion_charge',
            'cation_hydrated_radius_nm', 'anion_hydrated_radius_nm']
DYE_SOL  = ['MW', 'formal_charge']
STRUCT   = ['thickness_nm', 'flake_size_nm', 'dry_dspacing_nm']
XPS      = ['C-C', 'C-O', 'C=O'] 
SURFACE  = ['Zeta_potential_mV', 'Water_contact_angle']

# Feature sets A–D per solute type
FEAT_SETS = {
    'Salts': {
        'A': FILT + SALT_SOL,
        'B': FILT + SALT_SOL + STRUCT,
        'C': FILT + SALT_SOL + STRUCT + XPS,
        'D': FILT + SALT_SOL + STRUCT + XPS + SURFACE,
    },
    'Dyes': {
        'A': FILT + DYE_SOL,
        'B': FILT + DYE_SOL + STRUCT,
        'C': FILT + DYE_SOL + STRUCT + XPS,
        'D': FILT + DYE_SOL + STRUCT + XPS + SURFACE,
    },
}
D_FEATS = {
    'Salts': FILT + SALT_SOL + STRUCT + XPS + SURFACE,
    'Dyes':  FILT + DYE_SOL  + STRUCT + XPS + SURFACE,
}
# Feature set C (all rows, not restricted to D-subset) - used for CV + parity
FEATS_C = {
    'Salts': FILT + SALT_SOL + STRUCT + XPS,
    'Dyes':  FILT + DYE_SOL  + STRUCT + XPS,
}

print('Config loaded.')

In [ ]:
# --- Load data ---
df = pd.read_csv(DATA_FILE, encoding='utf-8-sig')
all_feats = FILT + SALT_SOL + DYE_SOL + STRUCT + XPS + SURFACE
for c in all_feats + [TARGET]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')
print(f'Loaded: {len(df)} rows')

In [ ]:
# --- Helper functions ---
def make_cb():
    return CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6,
                             verbose=False, random_seed=42)
def make_ridge():
    return make_pipeline(StandardScaler(), Ridge(alpha=10))

def prepare_D(df, d_feats):
    sub = df.dropna(subset=d_feats + [TARGET]).copy()
    sub = sub.drop_duplicates(subset=['doi'] + d_feats + [TARGET])
    return sub.reset_index(drop=True)

def prepare_C(df, feats):
    sub = df.dropna(subset=feats + [TARGET]).copy()
    sub = sub.drop_duplicates(subset=['doi'] + feats + [TARGET])
    return sub.reset_index(drop=True)

def loao(sub, feats, model_fn=make_cb):
    preds = np.full(len(sub), np.nan)
    for doi in sub['doi'].unique():
        tm = sub['doi'] != doi; vm = sub['doi'] == doi
        if tm.sum() < MIN_TRAIN: continue
        m = model_fn()
        m.fit(sub.loc[tm, feats], sub.loc[tm, TARGET])
        preds[vm] = m.predict(sub.loc[vm, feats])
    return preds

def metrics(y, preds):
    v = ~np.isnan(preds)
    return (r2_score(y[v], preds[v]),
            np.sqrt(mean_squared_error(y[v], preds[v])),
            mean_absolute_error(y[v], preds[v]),
            int(v.sum()))

def cv_random(sub, feats, seed=SEED):
    preds = np.full(len(sub), np.nan)
    for tr_idx, te_idx in KFold(n_splits=K_FOLDS, shuffle=True,
                                random_state=seed).split(sub):
        if len(tr_idx) < MIN_TRAIN: continue
        m = make_cb()
        m.fit(sub.iloc[tr_idx][feats], sub.iloc[tr_idx][TARGET])
        preds[te_idx] = m.predict(sub.iloc[te_idx][feats])
    return preds

def cv_gkf(sub, feats, seed=SEED):
    groups = sub['doi'].astype(str) + '__' + sub['membrane_uid'].astype(str)
    unique_g = list(groups.unique())
    rng = np.random.default_rng(seed); rng.shuffle(unique_g)
    g2fold   = {g: i % K_FOLDS for i, g in enumerate(unique_g)}
    fold_ids = groups.map(g2fold).values
    preds = np.full(len(sub), np.nan)
    for f in range(K_FOLDS):
        tm = fold_ids != f; vm = fold_ids == f
        if tm.sum() < MIN_TRAIN: continue
        m = make_cb()
        m.fit(sub.loc[tm, feats], sub.loc[tm, TARGET])
        preds[vm] = m.predict(sub.loc[vm, feats])
    return preds

def stats_box(ax, r2, rmse, n):
    ax.text(0.97, 0.05, f'$R^2$ = {r2:+.3f}\nRMSE = {rmse:.1f} pp\nn = {n}',
            transform=ax.transAxes, ha='right', va='bottom', fontsize=9.5,
            bbox=dict(boxstyle='round,pad=0.35', facecolor='white',
                      alpha=0.85, edgecolor='#cccccc'))

print('Functions defined.')

## Part 1 - Feature set A–D comparison (LOAO on D-subset)

To compare fairly across feature sets (A has fewer features than D),
all sets are evaluated on the **D-subset**: rows where all D features are available.

In [ ]:
print('Feature set comparison - CatBoost LOAO on D-subset')
print('=' * 65)
print(f'{"Solute":<8} {"Set":<4} {"n":>5} {"dois":>4} {"nf":>3}  '
      f'{"R²":>7}  {"RMSE (pp)":>9}  {"MAE (pp)":>8}')
print('-' * 65)

records_fs = []
for solute_name in ['Salts', 'Dyes']:
    typ = 'salt' if solute_name == 'Salts' else 'dye'
    raw = df[df['type'] == typ].copy()
    sub = prepare_D(raw, D_FEATS[solute_name])
    y   = sub[TARGET].values
    n, dois = len(sub), sub['doi'].nunique()
    print(f'\n  [{solute_name}]  n={n}, dois={dois}')

    for set_label, feats in FEAT_SETS[solute_name].items():
        preds = loao(sub, feats)
        r2, rmse, mae, n_pred = metrics(y, preds)
        print(f'  {solute_name:<8} {set_label:<4} {n:>5} {dois:>4} {len(feats):>3}  '
              f'{r2:>+7.3f}  {rmse:>9.2f}  {mae:>8.2f}')
        records_fs.append(dict(
            solute=solute_name, feature_set=set_label,
            n_features=len(feats), n_rows=n, n_dois=dois,
            R2_LOAO=round(r2, 4), RMSE_LOAO_pp=round(rmse, 3), MAE_LOAO_pp=round(mae, 3),
        ))

## Part 2 - CatBoost vs Ridge on feature set D (LOAO)

In [ ]:
MODELS = [('Ridge', make_ridge), ('CatBoost', make_cb)]

print('Ridge vs CatBoost - LOAO on D-subset')
print('=' * 65)
records_mc = []
for solute_name, feats in [('Salts', FEAT_SETS['Salts']['D']),
                            ('Dyes',  FEAT_SETS['Dyes']['D'])]:
    typ = 'salt' if solute_name == 'Salts' else 'dye'
    raw = df[df['type'] == typ].copy()
    sub = prepare_D(raw, D_FEATS[solute_name])
    y   = sub[TARGET].values
    n, dois = len(sub), sub['doi'].nunique()
    print(f'\n  [{solute_name}]  n={n}, dois={dois}')

    for model_name, model_fn in MODELS:
        preds = loao(sub, feats, model_fn)
        r2, rmse, mae, n_pred = metrics(y, preds)
        print(f'  {solute_name:<8} {model_name:<12} {n:>5} {dois:>4} {len(feats):>3}  '
              f'{r2:>+7.3f}  {rmse:>9.2f}  {mae:>8.2f}')
        records_mc.append(dict(
            solute=solute_name, model=model_name,
            n_rows=n, R2_LOAO=round(r2, 4),
            RMSE_LOAO_pp=round(rmse, 3), MAE_LOAO_pp=round(mae, 3),
        ))

## Part 3 - Feature set C: 3 CV strategies + parity plots

Using the **full dataset** (not restricted to D-subset) with feature set C.  
Parity plots use a single seed=42 run for Random and GroupKFold.

In [ ]:
SALT_COLORS = {
    'NaCl': '#1f77b4', 'Na₂SO₄': '#ff7f0e',
    'MgCl₂': '#2ca02c', 'MgSO₄': '#d62728', 'Others': '#9467bd',
}
MW_COLORS  = {'<400 Da': '#1f77b4', '400–700 Da': '#ff7f0e', '>700 Da': '#2ca02c'}
CH_MARKERS = {'+': '^', '−': 'v', '0': 'o'}
MW_ORDER   = ['<400 Da', '400–700 Da', '>700 Da']
CH_ORDER   = ['+', '−', '0']
CV_LABELS  = ['Random\n(seed = 42)', 'GroupKFold\n(by membrane, seed = 42)', 'LOAO']

def salt_group(row):
    t = (str(row.get('abbreviation') or '') + ' ' +
         str(row.get('solute_name') or '')).lower()
    if 'nacl' in t or 'sodium chloride' in t: return 'NaCl'
    if 'na2so4' in t or 'sodium sulfate' in t or 'sodium sulphate' in t: return 'Na₂SO₄'
    if 'mgcl2' in t or 'magnesium chloride' in t: return 'MgCl₂'
    if 'mgso4' in t or 'magnesium sulfate' in t or 'magnesium sulphate' in t: return 'MgSO₄'
    return 'Others'

def dye_mw_bin(mw):
    if pd.isna(mw): return 'unknown'
    if mw < 400: return '<400 Da'
    if mw < 700: return '400–700 Da'
    return '>700 Da'

def dye_charge_sign(fc):
    if pd.isna(fc) or fc == 0: return '0'
    return '+' if fc > 0 else '−'

print('Plot helpers defined.')

In [ ]:
for solute_name, typ, feats in [
    ('Salts', 'salt', FEATS_C['Salts']),
    ('Dyes',  'dye',  FEATS_C['Dyes']),
]:
    raw = df[df['type'] == typ].copy()
    sub = prepare_C(raw, feats)
    y   = sub[TARGET].values
    n_membranes = (sub['doi'].astype(str) + '__' + sub['membrane_uid'].astype(str)).nunique()
    print(f'\n=== {solute_name}: n={len(sub)}, dois={sub["doi"].nunique()}, '
          f'membranes={n_membranes} ===')

    if solute_name == 'Salts':
        sub['group'] = sub.apply(salt_group, axis=1)
    else:
        sub['mw_bin']      = sub['MW'].apply(dye_mw_bin)
        sub['charge_sign'] = sub['formal_charge'].apply(dye_charge_sign)

    p_rand = cv_random(sub, feats)
    p_gkf  = cv_gkf(sub, feats)
    p_loao = loao(sub, feats)
    pred_sets = [p_rand, p_gkf, p_loao]

    for label, p in zip(['Random', 'GKF', 'LOAO'], pred_sets):
        r2, rmse, mae, n_p = metrics(y, p)
        print(f'  {label:<8}: R²={r2:+.3f}  RMSE={rmse:.2f} pp  n={n_p}')

    # Parity plot
    all_vals = np.concatenate([y] + [p[~np.isnan(p)] for p in pred_sets])
    lo = max(0, float(all_vals.min()) - 5)
    hi = min(100, float(all_vals.max()) + 5)

    fig, axes = plt.subplots(1, 3, figsize=(14, 5.2))
    fig.suptitle(f'{solute_name} rejection - Feature set C',
                 fontsize=13, fontweight='bold', y=1.01)

    for ax, preds, title in zip(axes, pred_sets, CV_LABELS):
        valid = ~np.isnan(preds)
        ax.plot([lo, hi], [lo, hi], color='#444444', lw=0.9, ls='--', zorder=1)
        ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
        ax.set_aspect('equal', adjustable='box')
        ax.grid(True, alpha=0.22, lw=0.7)
        ax.set_title(title, fontsize=11, pad=5)
        ax.set_xlabel('Actual rejection (%)', fontsize=10)

        if solute_name == 'Salts':
            for grp, col in SALT_COLORS.items():
                m = (sub['group'] == grp).values & valid
                if m.sum():
                    ax.scatter(y[m], preds[m], c=col, s=28, alpha=0.72,
                               edgecolors='none', zorder=3)
        else:
            for mwb in MW_ORDER:
                for ch in CH_ORDER:
                    m = ((sub['mw_bin'] == mwb) & (sub['charge_sign'] == ch)).values & valid
                    if m.sum():
                        ax.scatter(y[m], preds[m], c=MW_COLORS[mwb],
                                   marker=CH_MARKERS[ch], s=45, alpha=0.80,
                                   edgecolors='none', zorder=3)

        r2v, rmsev, _, nv = metrics(y, preds)
        stats_box(ax, r2v, rmsev, nv)

    axes[0].set_ylabel('Predicted rejection (%)', fontsize=10)

    handles = []
    if solute_name == 'Salts':
        for grp, col in SALT_COLORS.items():
            cnt = (sub['group'] == grp).sum()
            if cnt:
                handles.append(Line2D([0],[0], marker='o', color='w',
                                      markerfacecolor=col, markersize=8,
                                      label=f'{grp}  (n={cnt})'))
        fig.legend(handles=handles, loc='lower center', ncol=len(handles),
                   fontsize=9.5, framealpha=0.85, bbox_to_anchor=(0.5, -0.07))
    else:
        for mwb in MW_ORDER:
            for ch in CH_ORDER:
                cnt = ((sub['mw_bin'] == mwb) & (sub['charge_sign'] == ch)).sum()
                if cnt:
                    ch_label = {'+': 'cationic', '−': 'anionic', '0': 'neutral'}[ch]
                    handles.append(Line2D([0],[0], marker=CH_MARKERS[ch], color='w',
                                          markerfacecolor=MW_COLORS[mwb], markersize=8,
                                          label=f'{mwb}, {ch_label}  (n={cnt})'))
        fig.legend(handles=handles, loc='lower center', ncol=3,
                   fontsize=9.5, framealpha=0.85, bbox_to_anchor=(0.5, -0.14))

    plt.tight_layout()
    plt.show()